# 07_prep_img_dataset.ipynb

This notebook prepares an **image dataset** for the VLM router project by:

1. Loading the existing **router dataset splits** (train/val/test).
2. Extracting unique image keys: `image_bytes_hash`, `source_config`, `source_index`.
3. Fetching the corresponding images from **The Cauldron** using `check_data_utils.fetch_cauldron_image`.
4. Encoding each image as **PNG bytes** and storing them in a **single Parquet file**:
   - `cauldron_images.parquet` with columns:
     - `image_bytes_hash`
     - `source_config`
     - `source_index`
     - `image_png` (PNG-compressed bytes)

The Parquet file can be joined back to your router dataset when you need actual image data, without refetching from Hugging Face.

> ️ **IMPORTANT:** You may need to adjust the paths in the **Config** section below to match your local repo structure and file names.


## 1. Imports and Configuration

In [1]:
import os
import sys
import io
from pathlib import Path

import pandas as pd
from tqdm import tqdm
from PIL import Image


In [2]:

# ------------------------------------------------------------------
# Project / data root configuration
# ------------------------------------------------------------------
NOTEBOOK_DIR = Path.cwd()

# Heuristic: assume project root is two levels up from this notebook
PROJECT_ROOT = NOTEBOOK_DIR.parents[2] if len(NOTEBOOK_DIR.parents) > 1 else NOTEBOOK_DIR
print("NOTEBOOK_DIR:", NOTEBOOK_DIR)
print("PROJECT_ROOT:", PROJECT_ROOT)

NOTEBOOK_DIR: /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/code_base/which_vlm/artemis
PROJECT_ROOT: /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router


In [3]:
# Adjust DATA_ROOT to match your repo layout if needed
# For example, if you store parquet data under:
#   <project-root>/dataset/which_vlm_data/
DATA_ROOT = PROJECT_ROOT / "dataset" 
print("DATA_ROOT:", DATA_ROOT)

DATA_ROOT: /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/dataset


In [4]:
# Directory to store image parquet + optional PNG cache
IMAGE_DATA_DIR = DATA_ROOT /  "images"
IMAGE_DATA_DIR.mkdir(parents=True, exist_ok=True)
print("IMAGE_DATA_DIR:", IMAGE_DATA_DIR)


IMAGE_DATA_DIR: /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/dataset/images


In [5]:

# Path to the Parquet file that will store all image bytes
IMG_PARQUET_PATH = IMAGE_DATA_DIR / "cauldron_images.parquet"
print("IMG_PARQUET_PATH:", IMG_PARQUET_PATH)

# ------------------------------------------------------------------
# Router dataset paths (UPDATE THESE TO MATCH YOUR FILES)
# ------------------------------------------------------------------
# Example defaults — change filenames if yours differ
TRAIN_PATH = DATA_ROOT / "final_dataset" /  "router_pivot_dataset_train.parquet"
VAL_PATH   = DATA_ROOT / "final_dataset" /  "router_pivot_dataset_validation.parquet"
TEST_PATH  = DATA_ROOT / "final_dataset" /  "router_pivot_dataset_test.parquet"

print("TRAIN_PATH:", TRAIN_PATH)
print("VAL_PATH:  ", VAL_PATH)
print("TEST_PATH: ", TEST_PATH)

IMG_PARQUET_PATH: /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/dataset/images/cauldron_images.parquet
TRAIN_PATH: /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/dataset/final_dataset/router_pivot_dataset_train.parquet
VAL_PATH:   /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/dataset/final_dataset/router_pivot_dataset_validation.parquet
TEST_PATH:  /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/dataset/final_dataset/router_pivot_dataset_test.parquet


## 2. Import image loading helpers from `check_data_utils`

In [6]:
# We expect `check_data_utils.py` to be importable either as:
#   - imports.check_data_utils
#   - check_data_utils
# or directly from PROJECT_ROOT once added to sys.path.

try:
    from imports.check_data_utils import fetch_cauldron_image, DEFAULT_IMAGE_ROOT
    print("Imported fetch_cauldron_image, DEFAULT_IMAGE_ROOT from imports.check_data_utils")
except ImportError:
    try:
        from imports.check_data_utils import fetch_cauldron_image, DEFAULT_IMAGE_ROOT
        print("Imported fetch_cauldron_image, DEFAULT_IMAGE_ROOT from check_data_utils")
    except ImportError:
        # Last resort: add project root to sys.path and retry
        sys.path.append(str(PROJECT_ROOT))
        from imports.check_data_utils import fetch_cauldron_image, DEFAULT_IMAGE_ROOT
        print("Imported fetch_cauldron_image, DEFAULT_IMAGE_ROOT after adding PROJECT_ROOT to sys.path")

print("DEFAULT_IMAGE_ROOT:", DEFAULT_IMAGE_ROOT)

Imported fetch_cauldron_image, DEFAULT_IMAGE_ROOT from imports.check_data_utils
DEFAULT_IMAGE_ROOT: dataset/which_vlm_data/images/cauldron


## 3. Load router datasets (train / val / test)

In [7]:
# Helper to load a parquet file if it exists, tagging with split name
def load_split(path: Path, split_name: str) -> pd.DataFrame:
    if not path.exists():
        print(f"[WARN] {split_name} file not found at: {path}")
        return None
    df = pd.read_parquet(path)
    df["split"] = split_name
    print(f"Loaded {split_name}: {len(df):,} rows")
    return df

dfs = []
for p, split in [(TRAIN_PATH, "train"), (VAL_PATH, "val"), (TEST_PATH, "test")]:
    df_split = load_split(p, split)
    if df_split is not None:
        dfs.append(df_split)

if not dfs:
    raise FileNotFoundError(
        "No router dataset parquet files were found. "
        "Please update TRAIN_PATH / VAL_PATH / TEST_PATH to point to your existing files."
    )

full_df = pd.concat(dfs, ignore_index=True)
print("Combined dataset shape:", full_df.shape)
full_df.head()

Loaded train: 63,963 rows
Loaded val: 13,706 rows
Loaded test: 13,707 rows
Combined dataset shape: (91376, 224)


,sample_id,run_id,timestamp_utc,image_path,image_bytes_hash,prompt_raw,prompt_formatted,system_prompt,source_dataset,source_config,...,gemma_3_27b__semantic_f1_gen_statements,gemma_3_27b__semantic_f1_gt_statements,gemma_3_27b__semantic_f1_matches,gemma_3_27b__semantic_f1_labels,gemma_3_27b__glider_score,gemma_3_27b__glider_reasoning,gemma_3_27b__glider_highlight,gemma_3_27b__glider_raw_output,subset_split,split
0,ai2d_00000_45f9e7163ea99b4c,exp_20251127_132944,2025-11-27T18:30:57.474332,None,45f9e7163ea99b4c,Question: What do respiration and combustion g...,None,None,cauldron_ai2d,ai2d,...,None,None,None,None,4.0,- The model's output is semantically correct a...,"[B, carbon dioxide, respiration, combustion, c...",<reasoning>\n- The model's output is semantic...,train,train
1,ai2d_00001_0135592f21ea024d,exp_20251127_132944,2025-11-27T18:31:03.209660,None,0135592f21ea024d,"Question: From the given food web, name any tw...",None,None,cauldron_ai2d,ai2d,...,None,None,None,None,4.0,- The model's answer is semantically correct a...,"[Jack Rabbit, Jack Rabbit, herbivores, eats pl...",<reasoning>\n- The model's answer is semantic...,train,train
2,ai2d_00002_df53e5451d7476fe,exp_20251127_132944,2025-11-27T18:31:09.116058,None,df53e5451d7476fe,Question: Anatomy One of a series of long curv...,None,None,cauldron_ai2d,ai2d,...,None,None,None,None,4.0,"- The model's output ""D. ribs"" is semantically...","[D. ribs, long, curved bones, 12 paired, chest...","<reasoning>\n- The model's output ""D. ribs"" i...",train,train
3,ai2d_00003_49e0ce3c07c66e5c,exp_20251127_132944,2025-11-27T18:31:15.901068,None,49e0ce3c07c66e5c,Question: What process does this diagram portr...,None,None,cauldron_ai2d,ai2d,...,None,None,None,None,5.0,- The model's output correctly identifies the ...,"[Photosynthesis, plant, sunlight, carbon dioxi...",<reasoning>\n- The model's output correctly i...,train,train
4,ai2d_00005_a70e661f5c7ae68d,exp_20251127_132944,2025-11-27T18:31:27.716607,None,a70e661f5c7ae68d,Question: Which type of rock consists of molte...,None,None,cauldron_ai2d,ai2d,...,None,None,None,None,5.0,"- The model's answer is exactly correct, as it...","[Igneous Rocks, molten rock, volcano, magma, l...",<reasoning>\n- The model's answer is exactly ...,train,train


## 4. Inspect image-related columns

In [8]:
print("Available columns:")
print(sorted(full_df.columns))

Available columns:
['cauldron_image_asset', 'cauldron_lookup_key', 'deepseek_ocr__error_message', 'deepseek_ocr__estimated_cost_usd', 'deepseek_ocr__glider_highlight', 'deepseek_ocr__glider_raw_output', 'deepseek_ocr__glider_reasoning', 'deepseek_ocr__glider_score', 'deepseek_ocr__gt_answer_letter', 'deepseek_ocr__inference_max_tokens', 'deepseek_ocr__inference_temperature', 'deepseek_ocr__inference_top_p', 'deepseek_ocr__input_tokens', 'deepseek_ocr__is_correct', 'deepseek_ocr__is_refusal', 'deepseek_ocr__latency_ms', 'deepseek_ocr__model_id', 'deepseek_ocr__model_name', 'deepseek_ocr__ok', 'deepseek_ocr__output_tokens', 'deepseek_ocr__pred_answer_letter', 'deepseek_ocr__response_length_chars', 'deepseek_ocr__response_length_tokens', 'deepseek_ocr__response_parsed', 'deepseek_ocr__response_raw', 'deepseek_ocr__score_contains_gt', 'deepseek_ocr__score_exact_match', 'deepseek_ocr__score_exact_match_normalized', 'deepseek_ocr__score_f1', 'deepseek_ocr__score_gt_in_response', 'deepseek_oc

In [9]:
expected_cols = ["image_bytes_hash", "source_config", "source_index"]

missing = [c for c in expected_cols if c not in full_df.columns]
if missing:
    raise KeyError(
        f"Expected columns not found in router dataset: {missing}.\n"
        f"Available columns: {list(full_df.columns)}\n"
        "Update `expected_cols` or adapt the notebook to your actual schema."
    )

# Keep only rows that have all required image info
image_keys = full_df[expected_cols].dropna(subset=expected_cols)
print(f"Rows with non-null image info: {len(image_keys):,}")

# Deduplicate image keys to avoid fetching the same image many times
unique_images = image_keys.drop_duplicates().reset_index(drop=True)
print(f"Unique images (by hash+config+index): {len(unique_images):,}")
unique_images.head()

Rows with non-null image info: 91,376
Unique images (by hash+config+index): 91,376


,image_bytes_hash,source_config,source_index
0,45f9e7163ea99b4c,ai2d,0.0
1,0135592f21ea024d,ai2d,1.0
2,df53e5451d7476fe,ai2d,2.0
3,49e0ce3c07c66e5c,ai2d,3.0
4,a70e661f5c7ae68d,ai2d,5.0


## 5. Helper: convert PIL image → PNG bytes

In [10]:
def pil_to_png_bytes(img: Image.Image) -> bytes:
    """Convert a PIL Image into PNG-compressed bytes for Parquet storage."""
    with io.BytesIO() as buf:
        img.save(buf, format="PNG")
        return buf.getvalue()

## 6. Determine missing images (resume-friendly)

In [11]:
image_key_cols = ["image_bytes_hash", "source_config", "source_index"]

existing_images = None
to_fetch = unique_images.copy()

if IMG_PARQUET_PATH.exists():
    print(f"Found existing image parquet at: {IMG_PARQUET_PATH}")
    existing_images = pd.read_parquet(IMG_PARQUET_PATH)
    print(f"Existing image records: {len(existing_images):,}")

    # Identify which keys are already stored
    merged = unique_images.merge(
        existing_images[image_key_cols],
        on=image_key_cols,
        how="left",
        indicator=True,
    )

    mask_missing = merged["_merge"] == "left_only"
    to_fetch = merged.loc[mask_missing, image_key_cols].reset_index(drop=True)

    print(f"Images already stored: {len(unique_images) - len(to_fetch):,}")
    print(f"Images still to fetch: {len(to_fetch):,}")
else:
    print("No existing image parquet found; all images will be fetched.")
    print(f"Images to fetch: {len(to_fetch):,}")

to_fetch.head()

Found existing image parquet at: /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/dataset/images/cauldron_images.parquet
Existing image records: 91,376
Images already stored: 91,376
Images still to fetch: 0


,image_bytes_hash,source_config,source_index


## 7. Fetch images from Cauldron and encode as PNG bytes

In [ ]:
from datasets import load_dataset
from functools import lru_cache
from ipywidgets import IntProgress, Label, HBox, VBox
from IPython.display import display
import time

# -------------------------------
# 1. Local Cauldron cache config
# -------------------------------
CAULDRON_CACHE_DIR = DATA_ROOT / "cauldron_raw_cache"
CAULDRON_CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("CAULDRON_CACHE_DIR:", CAULDRON_CACHE_DIR)

try:
    # Same repo you already use in check_data_utils
    from imports.check_data_utils import CAULDRON_REPO
except ImportError:
    from check_data_utils import CAULDRON_REPO


In [ ]:

@lru_cache(maxsize=None)
def get_cauldron_split(source_config: str):
    """
    Download + cache the full Cauldron 'train' split for a given config once.
    Afterwards, reads are local (no per-image HTTP).
    """
    print(f"[Cauldron] Loading local split for config='{source_config}' ...")
    ds = load_dataset(
        CAULDRON_REPO,
        source_config,
        split="train",
        cache_dir=str(CAULDRON_CACHE_DIR),
        download_mode="reuse_dataset_if_exists",  # don't re-download if present
    )
    print(f"[Cauldron]   -> {len(ds):,} rows for config='{source_config}'")
    return ds

# Group rows we still need by config (so we can read each split once)
if len(to_fetch) == 0:
    print("No images left to fetch.")
    grouped = {}
else:
    grouped = {
        cfg: df.reset_index(drop=True)
        for cfg, df in to_fetch.groupby("source_config")
    }
    print("Configs to process:", list(grouped.keys()))


In [ ]:
from ipywidgets import IntProgress, Label, HBox, VBox
from IPython.display import display
import time

total_images = len(to_fetch)
image_key_cols = ["image_bytes_hash", "source_config", "source_index"]

if total_images == 0 or len(grouped) == 0:
    print("No new images to fetch – all keys already stored in the parquet.")
    new_images = pd.DataFrame(columns=image_key_cols + ["image_png"])
else:
    # Jupyter interactive progress widgets
    progress = IntProgress(value=0, min=0, max=total_images, description='0%', bar_style='')
    status_label = Label(value="Starting local fetch...")
    rate_label = Label(value="Rate: -- img/s")
    eta_label = Label(value="ETA: --")
    display(VBox([HBox([progress, status_label]), rate_label, eta_label]))

    rows_out = []
    start = time.time()
    done = 0

    # Loop over each config and its rows, reading from LOCAL cached splits
    for cfg, cfg_df in grouped.items():
        ds = get_cauldron_split(cfg)   # downloads once, then cached

        for _, row in cfg_df.iterrows():
            idx = int(row["source_index"])
            if idx < 0 or idx >= len(ds):
                raise IndexError(f"Index {idx} out of range for config '{cfg}'")

            sample = ds[idx]
            img = sample["images"][0]          # same as in check_data_utils
            png_bytes = pil_to_png_bytes(img)

            rows_out.append({
                "image_bytes_hash": row["image_bytes_hash"],
                "source_config": row["source_config"],
                "source_index": idx,
                "image_png": png_bytes,
            })

            # ---- progress updates ----
            done += 1
            progress.value = done
            progress.description = f"{(done / total_images) * 100:4.1f}%"

            elapsed = time.time() - start
            rate = done / elapsed if elapsed > 0 else 0
            rate_label.value = f"Rate: {rate:.2f} img/s"

            remaining = total_images - done
            eta_sec = remaining / rate if rate > 0 else 0
            eta_label.value = f"ETA: {eta_sec/60:.1f} min"

            status_label.value = f"Fetched {done} / {total_images}"

    status_label.value = "Fetch complete (local)!"
    progress.bar_style = 'success'

    new_images = pd.DataFrame(rows_out, columns=image_key_cols + ["image_png"])

print("Newly fetched images:", len(new_images))


In [ ]:
### Cleanup 

# import shutil

# DELETE_CAULDRON_CACHE = False  # set to True when you want to clean up

# if DELETE_CAULDRON_CACHE:
#     shutil.rmtree(CAULDRON_CACHE_DIR, ignore_errors=True)
#     print("Removed local Cauldron cache:", CAULDRON_CACHE_DIR)
# else:
#     print("Keeping local Cauldron cache so future runs are offline and fast.")


## 8. Save / update image Parquet file

In [ ]:
if existing_images is not None and len(existing_images) > 0:
    print("Merging existing and newly fetched images...")
    combined = pd.concat([existing_images, new_images], ignore_index=True)
else:
    combined = new_images.copy()

# Remove any accidental duplicates by key
combined = combined.drop_duplicates(subset=image_key_cols, keep="last").reset_index(drop=True)

print("Final image table size:", len(combined))
combined.to_parquet(IMG_PARQUET_PATH, index=False)
print(f"Saved updated image parquet to: {IMG_PARQUET_PATH}")

## 9. (Optional) Populate on-disk PNG cache (for direct file access)

This step is **optional** and only needed if you want local PNG files for each image,
in a layout like:

```text
DEFAULT_IMAGE_ROOT / <source_config> / <image_bytes_hash>.png
```

This can be useful if you have other code that expects images on disk.

> ️ This will consume more disk space, since you're storing **both** the Parquet image bytes
> and the PNG files. Skip this step if you're tight on storage and are happy with Parquet-only.


In [ ]:
# Set this to True if you want to materialize PNG files on disk
WRITE_PNG_CACHE = False

if WRITE_PNG_CACHE:
    print("Writing PNG cache under:", DEFAULT_IMAGE_ROOT)
    DEFAULT_IMAGE_ROOT.mkdir(parents=True, exist_ok=True)

    # Load latest combined image table (just to be safe)
    img_table = pd.read_parquet(IMG_PARQUET_PATH)
    print("Image table rows:", len(img_table))

    for _, row in tqdm(img_table.iterrows(), total=len(img_table), desc="Writing PNG cache"):
        image_hash    = row["image_bytes_hash"]
        source_config = row["source_config"]
        image_bytes   = row["image_png"]

        out_dir = DEFAULT_IMAGE_ROOT / source_config
        out_dir.mkdir(parents=True, exist_ok=True)

        out_path = out_dir / f"{image_hash}.png"
        if out_path.exists():
            continue

        img = Image.open(io.BytesIO(image_bytes))
        img.save(out_path, format="PNG")

    print("Done writing PNG cache.")
else:
    print("Skipping PNG cache creation. Set WRITE_PNG_CACHE = True to enable.")

## 10. Quick sanity check: load an image back from Parquet

In [ ]:
# Reload parquet and display the first image to confirm everything works
img_table = pd.read_parquet(IMG_PARQUET_PATH)
print("Image table rows:", len(img_table))
img_table.head()


In [ ]:
# Only run this in an environment that can display images (e.g., Jupyter Lab)
from IPython.display import display

if len(img_table) > 0:
    sample_row = img_table.iloc[0]
    img = Image.open(io.BytesIO(sample_row["image_png"]))
    print("Sample image key:")
    print({
        "image_bytes_hash": sample_row["image_bytes_hash"],
        "source_config": sample_row["source_config"],
        "source_index": sample_row["source_index"],
    })
    display(img)
else:
    print("No rows in image parquet – nothing to display.")